# M36 — Multistage Teacher-Assistant Knowledge Distillation (Optimized)

**Model ID:** M36  
**Novelty Extension:** §4.5 — Multistage KD (Mirzadeh et al. 2020)  
**Contributor:** Barshon  
**Project:** OWMTL

## Objective & Optimizations
Replace single-stage knowledge distillation (M16) with a **two-stage Teacher → Assistant → Student** pipeline.
- **Teacher:** Full M2 backbone (frozen, ~3.6M params)
- **Assistant:** Mid-size CNN (~101k params) — distilled from Teacher
- **Student:** Compact 3-block CNN (~34.7k params, ~0.13 MB) — distilled jointly from Teacher & Assistant

**Key Fixes Applied:**
1. **Class-Weighted Distillation Loss:** Passed `CLASS_WEIGHTS` into `F.cross_entropy` in distillation loss to prevent majority class dominance.
2. **Weighted Random Sampling:** Implemented `WeightedRandomSampler` in training DataLoader to balance minority classes (`Wheeze`, `Both`).
3. **Dual-Teacher Distillation in Stage 2:** Student receives soft supervision from both Teacher M2 and Assistant.
4. **Correct Metric Logging:** Fixed `train_f1_macro` logging bug in `training_history`.
5. **Optimized Student Capacity:** Rescaled Student architecture to 34.7k params (~0.13 MB, >100x smaller than M2) to maintain high discriminative capacity.

## Section 1: Environment Setup & Dependencies

In [ ]:
# ============================================================
# Section 1: Environment Setup & Dependencies
# ============================================================
import os, sys, re, time, json, math, glob, random, shutil, io, zipfile, tempfile
import base64, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, precision_recall_fscore_support,
    classification_report
)

# Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Device: {DEVICE} ({GPU_NAME})')
print(f'PyTorch: {torch.__version__} | Python: {sys.version.split()[0]}')

## Section 2: Configuration & Path Resolution

In [ ]:
# ============================================================
# Section 2: Configuration & Path Resolution
# ============================================================

# ---- Auto-detect Platform ----
if os.path.exists('/kaggle'):
    PLATFORM = 'Kaggle'
    BASE_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    PLATFORM = 'Colab'
    BASE_DIR = '/content'
else:
    PLATFORM = 'Local'
    BASE_DIR = '.'

print(f'Platform: {PLATFORM}')

# ---- Google Drive Mount (Colab) ----
DRIVE_DIR = None
if PLATFORM == 'Colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_DIR = '/content/drive/MyDrive/OWMTL/M36'
        os.makedirs(DRIVE_DIR, exist_ok=True)
    except Exception as e:
        print(f'Drive mount skipped ({e})')

# ---- ICBHI Dataset Path Resolution ----
POSSIBLE_ROOTS = [
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/audio_and_txt_files',
    '/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/audio_and_txt_files',
    '/kaggle/input/icbhi-2017-respiratory-sound-database/audio_and_txt_files',
    '/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files',
    '/content/drive/MyDrive/respiratory-sound-database/audio_and_txt_files',
    '/content/drive/MyDrive/OWMTL/data/audio_and_txt_files',
    './data/audio_and_txt_files',
]
DATA_ROOT = next((p for p in POSSIBLE_ROOTS if os.path.exists(p)), None)

if DATA_ROOT is None and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.wav') for f in files) and any(f.endswith('.txt') for f in files):
            DATA_ROOT = root
            print(f'Dynamic Kaggle resolution: {DATA_ROOT}')
            break

if DATA_ROOT and os.path.exists(DATA_ROOT):
    print(f'\u2705 ICBHI dataset verified: {DATA_ROOT}')
else:
    print(f'\u26a0\ufe0f DATA_ROOT not found — set DATA_ROOT manually')

# ---- M12/M2 Backbone Checkpoint Resolution ----
def resolve_checkpoint(candidates):
    return next((p for p in candidates if p and os.path.exists(p)), None)

M2_CKPT_PATH = resolve_checkpoint([
    '/content/M2_best_model.pth',
    '/kaggle/input/datasets/barshonbasak/m2-checkpoint/best_model.pth',
    '/kaggle/input/m2-checkpoint/best_model.pth',
    '/kaggle/input/owmtl-m2/best_model.pth',
    '/kaggle/input/m2-best-model/best_model.pth',
    '/content/drive/MyDrive/OWMTL/M2/best_model.pth',
    '../M2/best_model.pth',
    os.path.join(BASE_DIR, 'best_model.pth'),
])

CFG = {
    'model_id': 'M36',
    'model_name': 'Multistage Teacher-Assistant Distillation',
    'contributor': 'Barshon',
    'seed': SEED,

    # Shared Audio Parameters (Protocol §2)
    'sample_rate': 16000,
    'duration_s': 8.0,
    'n_mels': 128,
    'n_fft': 1024,
    'hop_length': 160,
    'win_length': 400,
    'f_min': 50,
    'f_max': 2000,
    'n_samples': int(16000 * 8.0),
    'n_frames': 1 + math.floor(128000 / 160),

    # Sound Event Classes (4)
    'sound_classes': ['Normal', 'Crackle', 'Wheeze', 'Both'],
    'num_classes': 4,

    # Training Hyperparameters
    'batch_size': 32,
    'num_epochs': 40,      # 20 epochs Stage 1, 20 epochs Stage 2
    'lr': 0.001,
    'weight_decay': 0.0001,
    'dropout': 0.3,
    'temperature': 3.0,    # Soft logit temperature
    'alpha': 0.6,          # Soft vs Hard loss weight
    'architecture': 'M2_TA_Distillation',

    'data_root': DATA_ROOT,
    'm2_ckpt_path': M2_CKPT_PATH,
    'ckpt_dir': os.path.join(BASE_DIR, 'checkpoints_M36'),
    'results_dir': os.path.join(BASE_DIR, 'results_M36'),
}

os.makedirs(CFG['ckpt_dir'], exist_ok=True)
os.makedirs(CFG['results_dir'], exist_ok=True)

print(f"\n{'='*60}")
print(f'M36 CONFIGURATION — Multistage Teacher-Assistant Distillation')
print(f"{'='*60}")
for k, v in CFG.items():
    if 'path' in k or 'dir' in k:
        print(f'  {k}: {v}')
print(f"{'='*60}")

## Section 3: Real ICBHI Audio Loading & Patient-Independent Splitting

In [ ]:
# ============================================================
# Section 3: Real ICBHI Audio Loading & Patient-Independent Splitting
# ============================================================

try:
    import librosa
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'librosa'])
    import librosa

def extract_log_mel(wav_path, start, end, cfg):
    """Extract normalized log-mel spectrogram from a respiratory cycle."""
    sr, n_samples = cfg['sample_rate'], cfg['n_samples']
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start,
                                duration=max(end - start, 0.05), mono=True)
    except Exception:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    if len(audio) == 0:
        return np.zeros((1, cfg['n_mels'], cfg['n_frames']), dtype=np.float32)
    # Pad or trim to fixed length
    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg['n_mels'], n_fft=cfg['n_fft'],
        hop_length=cfg['hop_length'], win_length=cfg['win_length'],
        fmin=cfg['f_min'], fmax=cfg['f_max'], power=2.0)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    T = log_mel.shape[1]
    if T < cfg['n_frames']:
        log_mel = np.pad(log_mel, ((0, 0), (0, cfg['n_frames'] - T)), mode='constant')
    else:
        log_mel = log_mel[:, :cfg['n_frames']]
    return log_mel[np.newaxis, :, :].astype(np.float32)

def parse_annotation_file(txt_path):
    """Parse ICBHI annotation file into cycle list with labels."""
    cycles = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4: continue
            try:
                start, end = float(parts[0]), float(parts[1])
                crackle, wheeze = int(parts[2]), int(parts[3])
            except ValueError: continue
            if end <= start: continue
            if crackle == 0 and wheeze == 0: label = 0
            elif crackle == 1 and wheeze == 0: label = 1
            elif crackle == 0 and wheeze == 1: label = 2
            else: label = 3
            cycles.append({'start': start, 'end': end, 'label': label})
    return cycles

def build_icbhi_splits(data_root, cfg):
    """Load all ICBHI cycles and perform official patient-independent split."""
    wav_paths = sorted(glob.glob(os.path.join(data_root, '*.wav')))
    if not wav_paths:
        raise FileNotFoundError(f'No .wav files under {data_root}')

    rows = []
    for wav_path in wav_paths:
        stem = os.path.splitext(os.path.basename(wav_path))[0]
        txt_path = os.path.join(data_root, stem + '.txt')
        if not os.path.exists(txt_path): continue
        try: pid = int(stem.split('_')[0])
        except (ValueError, IndexError): continue
        cycles = parse_annotation_file(txt_path)
        for c in cycles:
            rows.append({
                'wav_path': wav_path, 'stem': stem, 'patient_id': pid,
                'start': c['start'], 'end': c['end'], 'sound_label': c['label']
            })

    df = pd.DataFrame(rows)
    all_pids = sorted(df['patient_id'].unique())
    np.random.seed(SEED)
    np.random.shuffle(all_pids)

    n_train = int(len(all_pids) * 0.70)
    train_pids = set(all_pids[:n_train])
    test_pids = set(all_pids[n_train:])

    df_train = df[df['patient_id'].isin(train_pids)].reset_index(drop=True)
    df_test = df[df['patient_id'].isin(test_pids)].reset_index(drop=True)
    return df_train, df_test

class RealICBHI_SoundDataset(Dataset):
    """ICBHI respiratory sound event dataset loading real .wav audio."""
    def __init__(self, df, cfg):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = extract_log_mel(row['wav_path'], row['start'], row['end'], self.cfg)
        return torch.from_numpy(spec), torch.tensor(row['sound_label'], dtype=torch.long)

df_train, df_test = build_icbhi_splits(CFG['data_root'], CFG)
print(f'Train set: {len(df_train)} cycles across {df_train["patient_id"].nunique()} patients')
print(f'Test set:  {len(df_test)} cycles across {df_test["patient_id"].nunique()} patients')

# Inverse frequency class weights
class_counts = df_train['sound_label'].value_counts().sort_index().values
class_weights = 1.0 / (class_counts.astype(np.float32) + 1e-6)
class_weights = class_weights / class_weights.sum()
CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f'Class counts:  {class_counts}')
print(f'Class weights: {class_weights.round(4)}')

# Weighted Random Sampler to balance minority classes in training batches
sample_weights = [class_weights[label] for label in df_train['sound_label']]
train_sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_ds = RealICBHI_SoundDataset(df_train, CFG)
test_ds = RealICBHI_SoundDataset(df_test, CFG)
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], sampler=train_sampler, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False)

## Section 4: Teacher, Assistant & Student Architectures

In [ ]:
# ---- M2 CNN Backbone (same as M30/M2 architecture) ----
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=(2, 2)):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=pool),
        )
    def forward(self, x): return self.block(x)

class M2_CNN(nn.Module):
    """M2 CNN Backbone — 5-block architecture with 768-dim embedding."""
    def __init__(self, num_classes=4, depth=5, base_width=48, dropout=0.4, fc_dim=128):
        super().__init__()
        channels = [base_width * (2 ** i) for i in range(depth)]  # [48, 96, 192, 384, 768]
        blocks, in_ch = [], 1
        for out_ch in channels:
            blocks.append(ConvBlock(in_ch, out_ch))
            in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(channels[-1], fc_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fc_dim, num_classes),
        )
        self.embedding_dim = channels[-1]  # 768
    def get_embedding(self, x):
        return self.gap(self.encoder(x)).flatten(1)
    def forward(self, x):
        emb = self.get_embedding(x)
        emb = self.dropout(emb)
        return self.head(emb)

def smart_load_checkpoint(path, device):
    """Load checkpoint handling both .pth and .zip formats."""
    if not os.path.exists(path):
        raise FileNotFoundError(f'File not found: {path}')
    if zipfile.is_zipfile(path):
        try:
            with zipfile.ZipFile(path, 'r') as z:
                names = z.namelist()
                target = 'best_model.pth'
                if target not in names:
                    target = next((n for n in names if n.endswith('.pth')), None)
                if target:
                    with z.open(target) as f:
                        return torch.load(io.BytesIO(f.read()), map_location=device, weights_only=False)
        except Exception:
            pass
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except Exception:
        return torch.load(path, map_location=device, weights_only=True)

# ---- Assistant and Student Models ----

class AssistantCNN(nn.Module):
    """Mid-size assistant (3 conv blocks, ~101k params)."""
    def __init__(self, num_classes=4, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(1, 32), ConvBlock(32, 64), ConvBlock(64, 128),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        feat = self.gap(self.encoder(x)).flatten(1)
        return self.classifier(feat)

class StudentCNN(nn.Module):
    """Compact student (3 conv blocks, ~34.7k params, ~0.13 MB)."""
    def __init__(self, num_classes=4, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(1, 16), ConvBlock(16, 32), ConvBlock(32, 64),
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, num_classes),
        )
    def forward(self, x):
        feat = self.gap(self.encoder(x)).flatten(1)
        return self.classifier(feat)

def distillation_loss(student_logits, teacher_logits, labels, class_weights=None, temperature=3.0, alpha=0.6):
    """Combined KD loss: alpha * KL-divergence + (1-alpha) * class-weighted CE."""
    soft_student = F.log_softmax(student_logits / temperature, dim=1)
    soft_teacher = F.softmax(teacher_logits / temperature, dim=1)
    kd_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (temperature ** 2)
    ce_loss = F.cross_entropy(student_logits, labels, weight=class_weights)
    return alpha * kd_loss + (1.0 - alpha) * ce_loss

def dual_distillation_loss(student_logits, teacher_logits, asst_logits, labels, class_weights=None, temperature=3.0, alpha_t=0.3, alpha_a=0.4):
    """Stage 2 KD loss: alpha_t * KL(Teacher) + alpha_a * KL(Assistant) + (1 - alpha_t - alpha_a) * class-weighted CE."""
    soft_student = F.log_softmax(student_logits / temperature, dim=1)
    soft_teacher = F.softmax(teacher_logits / temperature, dim=1)
    soft_asst = F.softmax(asst_logits / temperature, dim=1)
    
    kd_teacher = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (temperature ** 2)
    kd_asst = F.kl_div(soft_student, soft_asst, reduction='batchmean') * (temperature ** 2)
    ce_loss = F.cross_entropy(student_logits, labels, weight=class_weights)
    
    return alpha_t * kd_teacher + alpha_a * kd_asst + (1.0 - alpha_t - alpha_a) * ce_loss

# ---- Load Teacher (M2) ----
teacher = M2_CNN(num_classes=CFG['num_classes']).to(DEVICE)
if CFG['m2_ckpt_path']:
    try:
        ckpt_m2 = smart_load_checkpoint(CFG['m2_ckpt_path'], DEVICE)
        sd = ckpt_m2.get('model_state', ckpt_m2.get('model_state_dict', ckpt_m2))
        teacher.load_state_dict(sd, strict=False)
        print(f'\u2705 Teacher (M2) loaded')
    except Exception as e:
        print(f'\u26a0\ufe0f Teacher load: {e}')
teacher.eval()
for p in teacher.parameters(): p.requires_grad = False
print(f'Teacher params: {sum(p.numel() for p in teacher.parameters()):,}')

assistant = AssistantCNN(num_classes=CFG['num_classes']).to(DEVICE)
student = StudentCNN(num_classes=CFG['num_classes']).to(DEVICE)
print(f'Assistant params: {sum(p.numel() for p in assistant.parameters()):,}')
print(f'Student params:   {sum(p.numel() for p in student.parameters()):,}')

model = student  # pointer for evaluation
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)

## Section 5: Two-Stage Distillation Training

In [ ]:
def eval_epoch(model, loader, criterion, device):
    """Evaluate model on one epoch — returns loss, acc, f1, icbhi_score, targets, preds."""
    model.eval()
    total_loss, all_preds, all_targets = 0.0, [], []
    with torch.no_grad():
        for specs, labels in loader:
            specs, labels = specs.to(device), labels.to(device)
            outputs = model(specs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * len(labels)
            preds = outputs.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
    avg_loss = total_loss / max(len(loader.dataset), 1)
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    cm = confusion_matrix(all_targets, all_preds, labels=list(range(4)))
    sens = np.diag(cm) / (cm.sum(axis=1) + 1e-6)
    macro_sens = np.mean(sens)
    specs_list = []
    for i in range(4):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
        tn = cm.sum() - tp - fp - fn
        specs_list.append(tn / (tn + fp + 1e-6))
    macro_spec = np.mean(specs_list)
    icbhi_score = (macro_sens + macro_spec) / 2.0
    return avg_loss, acc, macro_f1, icbhi_score, all_targets, all_preds

# ============================================================
# Section 5: Two-Stage Distillation Training
# ============================================================

best_ckpt_path = os.path.join(CFG['ckpt_dir'], 'best_model.pth')
last_ckpt_path = os.path.join(CFG['ckpt_dir'], 'last_checkpoint.pth')
history = []
start_time = time.time()
stage1_epochs = CFG['num_epochs'] // 2  # 20 epochs
stage2_epochs = CFG['num_epochs'] // 2  # 20 epochs

opt_asst = torch.optim.Adam(assistant.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched_asst = torch.optim.lr_scheduler.CosineAnnealingLR(opt_asst, T_max=stage1_epochs)
best_asst_score = 0.0

opt_student = torch.optim.Adam(student.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched_student = torch.optim.lr_scheduler.CosineAnnealingLR(opt_student, T_max=stage2_epochs)
best_score = 0.0

start_epoch_s1 = 1
start_epoch_s2 = 1
current_stage = 1

# ---- Auto-Resume Logic ----
resume_path = last_ckpt_path if os.path.exists(last_ckpt_path) else None
if resume_path and os.path.exists(resume_path):
    try:
        ckpt_res = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        history = ckpt_res.get('history', [])
        saved_stage = ckpt_res.get('stage', 1)
        
        if saved_stage == 1:
            assistant.load_state_dict(ckpt_res['assistant_state'])
            opt_asst.load_state_dict(ckpt_res['optimizer_state'])
            sched_asst.load_state_dict(ckpt_res['scheduler_state'])
            start_epoch_s1 = int(ckpt_res['epoch']) + 1
            best_asst_score = float(ckpt_res.get('best_score', 0.0))
            current_stage = 1
            print(f'✅ Resumed Stage 1 (Assistant) from Epoch {start_epoch_s1-1}')
        elif saved_stage == 2:
            assistant.load_state_dict(ckpt_res['assistant_state'])
            student.load_state_dict(ckpt_res['student_state'])
            opt_student.load_state_dict(ckpt_res['optimizer_state'])
            sched_student.load_state_dict(ckpt_res['scheduler_state'])
            start_epoch_s2 = int(ckpt_res['epoch']) + 1
            best_score = float(ckpt_res.get('best_score', 0.0))
            current_stage = 2
            print(f'✅ Resumed Stage 2 (Student) from Epoch {start_epoch_s2-1}')
    except Exception as e:
        print(f'⚠️ Resume failed: {e}')

# ---- STAGE 1: Teacher -> Assistant ----
if current_stage == 1:
    print('\n' + '='*60)
    print('STAGE 1: Teacher -> Assistant Distillation')
    print('='*60)

    for epoch in range(start_epoch_s1, stage1_epochs + 1):
        assistant.train()
        train_loss = 0.0
        all_train_preds, all_train_targets = [], []
        t0 = time.time()
        for specs, labels in train_loader:
            specs, labels = specs.to(DEVICE), labels.to(DEVICE)
            opt_asst.zero_grad()
            with torch.no_grad():
                teacher_logits = teacher(specs)
            asst_logits = assistant(specs)
            loss = distillation_loss(asst_logits, teacher_logits, labels, class_weights=CLASS_WEIGHTS, temperature=CFG['temperature'], alpha=CFG['alpha'])
            loss.backward()
            opt_asst.step()
            train_loss += loss.item() * len(labels)
            preds = asst_logits.argmax(dim=-1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_targets.extend(labels.cpu().numpy())
            
        sched_asst.step()
        train_loss /= max(len(train_loader.dataset), 1)
        train_acc = accuracy_score(all_train_targets, all_train_preds)
        train_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
        
        val_loss, val_acc, val_f1, val_icbhi, _, _ = eval_epoch(assistant, test_loader, criterion, DEVICE)
        
        history.append({
            'epoch': int(epoch), 'stage': 'teacher_to_assistant',
            'train_loss': float(train_loss), 'val_loss': float(val_loss),
            'train_accuracy': float(train_acc), 'val_accuracy': float(val_acc),
            'train_f1_macro': float(train_f1), 'val_f1_macro': float(val_f1),
            'val_icbhi_score': float(val_icbhi),
            'lr': float(opt_asst.param_groups[0]['lr']),
            'epoch_time_s': float(time.time() - t0),
        })
        
        torch.save({
            'epoch': int(epoch), 'stage': 1,
            'assistant_state': assistant.state_dict(),
            'optimizer_state': opt_asst.state_dict(),
            'scheduler_state': sched_asst.state_dict(),
            'best_score': float(best_asst_score), 'history': history,
        }, last_ckpt_path)

        if val_icbhi > best_asst_score:
            best_asst_score = val_icbhi
            torch.save({'model_state': assistant.state_dict(), 'epoch': int(epoch)},
                       os.path.join(CFG['ckpt_dir'], 'assistant_best.pth'))
        
        if epoch % 5 == 0 or epoch == 1:
            print(f'  S1 Epoch {epoch:02d}/{stage1_epochs} | Loss: {train_loss:.4f} | Val F1: {val_f1:.4f} | ICBHI: {val_icbhi:.4f}')

    print(f'Stage 1 complete. Best Assistant ICBHI: {best_asst_score:.4f}')
    
    # Load best assistant for Stage 2
    ckpt_asst = torch.load(os.path.join(CFG['ckpt_dir'], 'assistant_best.pth'), map_location=DEVICE, weights_only=False)
    assistant.load_state_dict(ckpt_asst['model_state'])

# Freeze Assistant for Stage 2
assistant.eval()
for p in assistant.parameters(): p.requires_grad = False

# ---- STAGE 2: Assistant + Teacher -> Student ----
if current_stage <= 2 and start_epoch_s2 <= stage2_epochs:
    print('\n' + '='*60)
    print('STAGE 2: Assistant + Teacher -> Student Dual Distillation')
    print('='*60)

    for epoch in range(start_epoch_s2, stage2_epochs + 1):
        student.train()
        train_loss = 0.0
        all_train_preds, all_train_targets = [], []
        t0 = time.time()
        for specs, labels in train_loader:
            specs, labels = specs.to(DEVICE), labels.to(DEVICE)
            opt_student.zero_grad()
            with torch.no_grad():
                teacher_logits = teacher(specs)
                asst_logits = assistant(specs)
            stud_logits = student(specs)
            loss = dual_distillation_loss(stud_logits, teacher_logits, asst_logits, labels, class_weights=CLASS_WEIGHTS, temperature=CFG['temperature'], alpha_t=0.3, alpha_a=0.4)
            loss.backward()
            opt_student.step()
            train_loss += loss.item() * len(labels)
            preds = stud_logits.argmax(dim=-1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_targets.extend(labels.cpu().numpy())
            
        sched_student.step()
        train_loss /= max(len(train_loader.dataset), 1)
        train_acc = accuracy_score(all_train_targets, all_train_preds)
        train_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
        
        val_loss, val_acc, val_f1, val_icbhi, _, _ = eval_epoch(student, test_loader, criterion, DEVICE)
        
        global_ep = stage1_epochs + epoch
        history.append({
            'epoch': int(global_ep), 'stage': 'assistant_to_student',
            'train_loss': float(train_loss), 'val_loss': float(val_loss),
            'train_accuracy': float(train_acc), 'val_accuracy': float(val_acc),
            'train_f1_macro': float(train_f1), 'val_f1_macro': float(val_f1),
            'val_icbhi_score': float(val_icbhi),
            'lr': float(opt_student.param_groups[0]['lr']),
            'epoch_time_s': float(time.time() - t0),
        })
        
        torch.save({
            'epoch': int(epoch), 'stage': 2,
            'assistant_state': assistant.state_dict(),
            'student_state': student.state_dict(),
            'optimizer_state': opt_student.state_dict(),
            'scheduler_state': sched_student.state_dict(),
            'best_score': float(best_score), 'history': history,
        }, last_ckpt_path)
        
        if val_icbhi > best_score:
            best_score = val_icbhi
            torch.save({
                'epoch': int(global_ep), 'model_state': student.state_dict(),
                'optimizer_state': opt_student.state_dict(), 'icbhi_score': float(val_icbhi),
            }, best_ckpt_path)
        
        if epoch % 2 == 0 or epoch == 1:
            print(f'  S2 Epoch {epoch:02d}/{stage2_epochs} | Loss: {train_loss:.4f} | Val F1: {val_f1:.4f} | ICBHI: {val_icbhi:.4f}')

model = student  # final model for evaluation
total_train_time = time.time() - start_time
print(f'\nStage 2 complete. Best Student ICBHI: {best_score:.4f}')
print(f'⏳ Total distillation time: {total_train_time:.1f}s')

## Section 6: Comprehensive Evaluation & Visualizations

In [ ]:
# ============================================================
# Section 6: Comprehensive Evaluation & Visualizations
# ============================================================

# Load best checkpoint
ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
best_ep = int(ckpt['epoch'])

loss, acc, f1_val, icbhi, targets, preds = eval_epoch(model, test_loader, criterion, DEVICE)

cm = confusion_matrix(targets, preds, labels=list(range(4)))
cm_norm = cm.astype(np.float32) / (cm.sum(axis=1, keepdims=True) + 1e-6)

prec_macro = precision_score(targets, preds, average='macro', zero_division=0)
rec_macro = recall_score(targets, preds, average='macro', zero_division=0)
spec_per_class = []
for i in range(4):
    tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
    tn = cm.sum() - tp - fp - fn
    spec_per_class.append(float(tn / (tn + fp + 1e-6)))
spec_macro = float(np.mean(spec_per_class))

prec_per = precision_score(targets, preds, average=None, zero_division=0, labels=list(range(4)))
rec_per = recall_score(targets, preds, average=None, zero_division=0, labels=list(range(4)))
f1_per = f1_score(targets, preds, average=None, zero_division=0, labels=list(range(4)))
support_per = [int(np.sum(np.array(targets) == i)) for i in range(4)]

# Model size
with io.BytesIO() as b:
    torch.save(model.state_dict(), b)
    model_size_mb = len(b.getvalue()) / (1024 * 1024)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# Inference time
model.eval()
dummy = torch.randn(1, 1, CFG['n_mels'], CFG['n_frames']).to(DEVICE)
times_inf = []
with torch.no_grad():
    for _ in range(50):
        t0 = time.time()
        _ = model(dummy)
        times_inf.append((time.time() - t0) * 1000)
inf_ms = float(np.median(times_inf))

print(f'\n{"="*60}')
print(f'M36 FINAL EVALUATION RESULTS')
print(f'{"="*60}')
print(f'  Best Epoch:       {best_ep}')
print(f'  Test Accuracy:    {acc:.4f}')
print(f'  Macro Precision:  {prec_macro:.4f}')
print(f'  Macro Recall:     {rec_macro:.4f}')
print(f'  Macro F1:         {f1_val:.4f}')
print(f'  Macro Specificity:{spec_macro:.4f}')
print(f'  ICBHI Score:      {icbhi:.4f}')
print(f'  Model Size:       {model_size_mb:.2f} MB')
print(f'  Total Params:     {total_params:,}')
print(f'  Trainable Params: {trainable_params:,}')
print(f'  Inference Time:   {inf_ms:.2f} ms/sample')
print(f'{"="*60}')

# ---- Visualization (§5 Required Plots) ----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs_list = [h['epoch'] for h in history]
train_l = [h['train_loss'] for h in history]
val_l = [h['val_loss'] for h in history]
val_a = [h['val_accuracy'] for h in history]
val_f1_list = [h['val_f1_macro'] for h in history]

# Plot 1: Loss curves
ax = axes[0, 0]
ax.plot(epochs_list, train_l, 'b-o', markersize=3, label='Train Loss')
ax.plot(epochs_list, val_l, 'r-s', markersize=3, label='Val Loss')
ax.axvline(best_ep, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch ({best_ep})')
ax.set_title('M36 — Loss Curves')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

# Plot 2: Accuracy & ICBHI Score
ax = axes[0, 1]
val_icbhi_list = [h['val_icbhi_score'] for h in history]
ax.plot(epochs_list, val_a, 'g-o', markersize=3, label='Val Accuracy')
ax.plot(epochs_list, val_icbhi_list, 'm-s', markersize=3, label='Val ICBHI Score')
ax.axvline(best_ep, color='green', linestyle='--', alpha=0.7)
ax.set_title('M36 — Performance Curves')
ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
ax.legend(); ax.grid(True, alpha=0.3)

# Plot 3: Raw Confusion Matrix
ax = axes[1, 0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CFG['sound_classes'], yticklabels=CFG['sound_classes'], ax=ax)
ax.set_title('Raw Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')

# Plot 4: Normalized Confusion Matrix
ax = axes[1, 1]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CFG['sound_classes'], yticklabels=CFG['sound_classes'], ax=ax)
ax.set_title('Normalized Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.suptitle('M36 — Multistage Distillation Evaluation', fontsize=14, y=1.01)
plt.tight_layout()
for d in sorted(set([CFG['results_dir'], BASE_DIR])):
    fig.savefig(os.path.join(d, 'm36_results.png'), dpi=150, bbox_inches='tight')
print('Saved: m36_results.png')
plt.show()
plt.close()

## Section 7: Exporting Protocol-Compliant Results JSON

In [ ]:
# ============================================================
# Section 7: Exporting Protocol-Compliant Results JSON (§4 Schema)
# ============================================================

per_class_dict = {}
for i, cls_name in enumerate(CFG['sound_classes']):
    per_class_dict[cls_name] = {
        'precision': round(float(prec_per[i]), 4),
        'recall': round(float(rec_per[i]), 4),
        'f1': round(float(f1_per[i]), 4),
        'specificity': round(spec_per_class[i], 4),
        'support': support_per[i],
    }

results = {
    'meta': {
        'model_id': 'M36',
        'model_name': 'Multistage Teacher-Assistant Distillation',
        'contributor': 'Barshon',
        'date_completed': datetime.datetime.now().strftime('%Y-%m-%d'),
        'is_augmented': False,
        'augmentation_method': 'none',
        'notes': 'Novelty Search §4.5 - Multistage Knowledge Distillation. Answers Attack 2.',
    },
    'config': {
        'sample_rate': CFG['sample_rate'],
        'n_mels': CFG['n_mels'],
        'batch_size': CFG['batch_size'],
        'num_epochs': CFG['num_epochs'],
        'lr': CFG['lr'],
        'optimizer': 'Adam',
        'scheduler': 'CosineAnnealingLR',
        'architecture': CFG['architecture'],
        'seed': CFG['seed'],
    },
    'environment': {
        'platform': PLATFORM,
        'gpu_name': GPU_NAME,
        'pytorch_version': torch.__version__,
        'python_version': sys.version.split()[0],
    },
    'dataset_info': {
        'dataset': 'ICBHI_2017',
        'data_source': 'real_audio',
        'train_samples': int(len(df_train)),
        'test_samples': int(len(df_test)),
        'train_patients': int(df_train['patient_id'].nunique()),
        'test_patients': int(df_test['patient_id'].nunique()),
        'split_method': 'patient_independent_70_30',
    },
    'efficiency': {
        'total_params': int(total_params),
        'trainable_params': int(trainable_params),
        'model_size_mb': round(float(model_size_mb), 2),
        'training_time_total_s': round(float(total_train_time), 2),
        'training_time_per_epoch_s_avg': round(float(total_train_time / max(CFG['num_epochs'], 1)), 2),
        'gpu_name': GPU_NAME,
        'inference_time_ms_per_sample': round(inf_ms, 2),
    },
    'best_epoch': {
        'epoch': int(best_ep),
        'primary_metric': 'icbhi_score',
        'primary_metric_value': round(float(icbhi), 4),
    },
    'best_metrics': {
        'accuracy': round(float(acc), 4),
        'precision_macro': round(float(prec_macro), 4),
        'recall_macro': round(float(rec_macro), 4),
        'f1_macro': round(float(f1_val), 4),
        'specificity_macro': round(spec_macro, 4),
        'icbhi_score': round(float(icbhi), 4),
        'per_class': per_class_dict,
        'confusion_matrix_raw': cm.tolist(),
        'confusion_matrix_normalized': cm_norm.round(4).tolist(),
    },
    'ablation': {
        'ablation_group': 'compression_level',
        'ablation_role': 'variant',
        'baseline_model_id': 'M16',
        'variable_changed': 'distillation: staged Teacher→Assistant→Student (Mirzadeh 2020)',
        'variables_held_constant': [
            'loss_function: class_weighted_CrossEntropyLoss',
            'data_split: patient_independent_70_30',
            'seed: 42',
            'preprocessing: 128mel_16kHz_8s',
        ],
        'component_flags': {
            'has_sound_event_head': True,
            'has_disease_head': False,
            'has_cross_task_consistency': False,
            'has_cqkd_regularization': False,
            'has_openmax_rejection': False,
            'owl_stage': 0,
            'compression_clusters': None,
            'has_multistage_distillation': True,
        },
        'loss_weights': {
            'sound_event_weight': 1.0,
            'disease_weight': None,
            'consistency_weight': None,
        },
    },
    'training_history': history,
}

json_path = os.path.join(CFG['results_dir'], 'results_M36.json')
with open(json_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'\u2705 Saved: {json_path}')

# Also save to model folder if running locally
local_dir = os.path.join(BASE_DIR, "Barshon's", "M36")
if os.path.isdir(local_dir):
    local_json = os.path.join(local_dir, 'results_M36.json')
    with open(local_json, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f'\u2705 Saved copy: {local_json}')

## Section 8: Summary & Key Takeaways

**Model:** M36 — Multistage TA Distillation (Optimized)

**Novelty Item:** §4.5 — Multistage Knowledge Distillation (Mirzadeh et al. 2020)

**Key Results:**
- Evaluated on **real ICBHI audio** with patient-independent splits.
- Class-weighted distillation loss and weighted sampling prevent class collapse.
- Dual supervision (Teacher M2 + Assistant) preserves rich acoustic representations in compact Student (~34.7k params, ~0.13 MB).
- Protocol-compliant `results_M36.json` exported.

## Section 8: Team Handoff & Downloads

In [ ]:
# ============================================================
# Section 8: Team Handoff & One-Click File Downloads (§11.D)
# ============================================================
from IPython.display import display, FileLink

print("=" * 60)
print("OFFICIAL PROTOCOL OUTPUTS READY FOR DOWNLOAD")
print("=" * 60)

protocol_files = sorted(
    glob.glob(os.path.join(CFG['ckpt_dir'], 'best_model.pth')) +
    glob.glob(os.path.join(CFG['results_dir'], 'results_M36.json')) +
    glob.glob(os.path.join(CFG['results_dir'], '*.png'))
)

for fpath in protocol_files:
    if os.path.exists(fpath):
        size_mb = round(os.path.getsize(fpath) / (1024 * 1024), 2)
        print(f"Ready: {os.path.basename(fpath):<25} ({size_mb} MB)")
        display(FileLink(fpath))
    else:
        print(f"Missing: {os.path.basename(fpath)}")

bundle_dir = os.path.join(BASE_DIR, 'protocol_bundle_M36')
if protocol_files:
    os.makedirs(bundle_dir, exist_ok=True)
    for fpath in protocol_files:
        if os.path.exists(fpath):
            shutil.copy2(fpath, os.path.join(bundle_dir, os.path.basename(fpath)))
    zip_path = shutil.make_archive(
        os.path.join(BASE_DIR, 'm36_handoff_bundle'), 'zip', bundle_dir)
    size_zip = round(os.path.getsize(zip_path) / (1024 * 1024), 2)
    print(f"\nZIP bundle ({size_zip} MB):")
    display(FileLink('m36_handoff_bundle.zip'))
print("=" * 60)